# 监督微调（2）模型训练

In [1]:
import pickle

from src.core import *
from src.gpt import *

In [2]:
np.random.seed(42)

In [3]:
class SFTDataset(Dataset):

    def __init__(self, filename, context_size=64, split=0.9):
        self.filename = filename
        self.context_size = context_size
        self.split = split
        super().__init__(1)

    def load(self):
        with open(self.filename, "rb") as f:
            examples = pickle.load(f)

        split = int(len(examples) * self.split)
        self.train_data = self._pack(examples[:split])
        self.test_data = self._pack(examples[split:])

    def _pack(self, examples):
        xs, ys, masks = [], [], []
        for prompt, response in examples:
            x, y, mask = self._build_example(prompt, response)
            xs.append(x)
            ys.append(y)
            masks.append(mask)
        return xs, ys, masks

    def _build_example(self, prompt, response):
        tokens = list(prompt) + list(response)
        if len(tokens) > self.context_size + 1:
            overflow = len(tokens) - (self.context_size + 1)
            prompt = prompt[overflow:] if overflow < len(prompt) else []
            tokens = list(prompt) + list(response)
            tokens = tokens[-(self.context_size + 1):]

        response_start = len(prompt)

        x = np.array(tokens[:-1], dtype=np.int64)
        y = np.array(tokens[1:], dtype=np.int64)
        mask = np.arange(len(y)) + 1 >= response_start
        return x, y, mask

    def __getitem__(self, index):
        s = slice(index * self.batch_size, (index + 1) * self.batch_size)
        x, y, mask = self.data
        return Tensor(x[s]), Tensor(y[s]), Tensor(mask[s])

In [4]:
class SFTLoss(Loss):

    def loss(self, p: Tensor, y: Tensor, mask):
        exp = np.exp(p.data - np.max(p.data, axis=-1, keepdims=True))
        softmax = exp / np.sum(exp, axis=-1, keepdims=True)

        flat_softmax = softmax.reshape(-1, softmax.shape[-1])
        flat_y = y.data.reshape(-1).astype(np.int64)
        flat_mask = mask.data.reshape(-1)

        rows = np.arange(len(flat_y))
        n = max(np.sum(flat_mask), 1.0)

        log = np.log(np.clip(flat_softmax[rows, flat_y], 1e-10, 1))
        ce = Tensor(0 - np.sum(log * flat_mask) / n)

        def gradient_fn():
            flat_grad = flat_softmax.copy()
            flat_grad[rows, flat_y] -= 1
            flat_grad *= flat_mask[:, None]
            p.grad += ce.grad * flat_grad.reshape(softmax.shape) / n

        return ce.attach(gradient_fn, parents={p})

In [5]:
class SFTModel(GPTModel):

    def train(self, dataset, epochs, scheduler=None, filename=None):
        self.layer.train()

        steps = 0
        for epoch in range(epochs):
            order = list(range(len(dataset)))
            np.random.shuffle(order)

            total_loss = 0.0
            for step, i in enumerate(order):
                if scheduler is not None:
                    self.optimizer.lr = scheduler.step(steps)

                feature, label, mask = dataset[i]
                prediction = self.layer(feature)
                loss = self.loss_fn(prediction, label, mask)

                self.optimizer.zero_grad()
                loss.backward()
                total_loss += float(loss.data)
                self.optimizer.clip_grad_norm()
                self.optimizer.step()
                steps += 1

                if (step + 1) % 100 == 0:
                    lr = f" lr {self.optimizer.lr:.6f}" if scheduler is not None else ""
                    print(f"epoch {epoch + 1} step {step + 1}/{len(dataset)} loss {(total_loss / 100):.4f}{lr}")
                    total_loss = 0.0

            if filename is not None:
                self.save(filename)
                print(f"epoch {epoch + 1} saved SFT model to {filename}")

    def test(self, dataset):
        return None

In [6]:
DATA_FILE = "../../../tinyshakespeare.txt"
MODEL_FILE = "../../../tinyshakespeare-gpt.npz"
SFT_SAMPLES = "../../sft-samples.pkl"
SFT_MODEL = "../../tinyshakespeare-sft.npz"

In [7]:
LEARNING_RATE = 0.0001
BATCH_SIZE = 4
CONTEXT_SIZE = 32
EMBEDDING_SIZE = 64
HEADS = 2
BLOCKS = 2

In [8]:
dataset = CharDataset(DATA_FILE, BATCH_SIZE, CONTEXT_SIZE)
layer = GPT(dataset.vocab_size, CONTEXT_SIZE, EMBEDDING_SIZE, HEADS, BLOCKS)
loss_fn = SFTLoss()
optimizer = AdamWOptimizer(layer.parameters, lr=LEARNING_RATE)
model = SFTModel(layer, loss_fn, optimizer)
model.load(MODEL_FILE)

In [9]:
sft_dataset = SFTDataset(SFT_SAMPLES, CONTEXT_SIZE)
scheduler = WarmupCosineScheduler(LEARNING_RATE, len(sft_dataset), 50, LEARNING_RATE / 10)
model.train(sft_dataset, 1, scheduler, SFT_MODEL)

epoch 1 step 100/921 loss 2.0043 lr 0.000099
epoch 1 step 200/921 loss 2.0054 lr 0.000094
epoch 1 step 300/921 loss 2.0260 lr 0.000083
epoch 1 step 400/921 loss 1.9685 lr 0.000069
epoch 1 step 500/921 loss 1.9782 lr 0.000053
epoch 1 step 600/921 loss 1.9132 lr 0.000037
epoch 1 step 700/921 loss 1.9108 lr 0.000024
epoch 1 step 800/921 loss 1.8877 lr 0.000014
epoch 1 step 900/921 loss 1.9586 lr 0.000010
epoch 1 saved SFT model to ../../tinyshakespeare-sft.npz


In [10]:
print(model.generate(dataset, prompt="ROMEO:"))

ROMEO:
O, I we save it cready, lives all and the as umhough roffor greated.

LUCESTER:
His, you.

ISABERLO:
No, man:
Hath persely, lards on
What frace men; glord; for King thy hour mone.
O Sull;
Are, yethouse in mer, my cantare is strence wath
Thing, and for Go hearmition you, God--

But RIOLIXE:
G Beare friend!
Which curse my the get strues not my bid not a drump?
If do your must not me would lord? Ret Tair:
The theree of n
this have; I way.
Vo's: Con's yet, yet our e'ep
Fire him ibe caurse Rich muster up.

BERT
